# Part 3: NLP and Sequence Modeling Mini Project

**Goal:** Build an NLP pipeline to classify customer support messages by sentiment  
**Dataset:** Customer Support Text Classification Dataset  
**Dataset Source:** [Google Drive – Part 3 Dataset](https://drive.google.com/drive/folders/1akV6po4Nrgkc3yQrJkzA6cJlV-wBvUYs?usp=sharing)


## Setup

In [ ]:
# Install required NLP libraries
!pip install nltk -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import os
import warnings
warnings.filterwarnings('ignore')

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (classification_report, confusion_matrix,
                              ConfusionMatrixDisplay, accuracy_score)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

os.makedirs('results', exist_ok=True)
print(f"TensorFlow: {tf.__version__}")
print("All libraries loaded successfully!")


## Task 1: Dataset Understanding

In [ ]:
# Load the dataset
df = pd.read_csv('customer_support_text_classification.csv')

print("=" * 55)
print("DATASET OVERVIEW")
print("=" * 55)
print(f"Total records : {df.shape[0]}")
print(f"Columns       : {list(df.columns)}")


In [ ]:
# Target label distribution
print("\n--- Sentiment Label Distribution ---")
print(df['sentiment_label'].value_counts())
print(f"\nChurn Rate (class balance):")
print(df['sentiment_label'].value_counts(normalize=True).mul(100).round(2).astype(str) + '%')


In [ ]:
# Sample text records
print("\n--- Sample Records ---")
for label in df['sentiment_label'].unique():
    sample = df[df['sentiment_label'] == label]['customer_message'].iloc[0]
    print(f"\n[{label.upper()}]\n{sample}")


In [ ]:
# Text length analysis
df['text_length'] = df['customer_message'].apply(len)
df['token_count'] = df['customer_message'].apply(lambda x: len(x.split()))

print("\n--- Text Length Statistics ---")
print(df[['text_length', 'token_count', 'word_count']].describe().round(2))

# Plot distributions
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Class distribution
counts = df['sentiment_label'].value_counts()
colors = ['#2ecc71', '#3498db', '#e74c3c']
bars = axes[0].bar(counts.index, counts.values, color=colors, edgecolor='black')
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
                 str(val), ha='center', fontweight='bold')
axes[0].set_title('Class Distribution', fontsize=13)
axes[0].set_ylabel('Count')

# Word count by sentiment
df.boxplot(column='token_count', by='sentiment_label', ax=axes[1],
           patch_artist=True)
axes[1].set_title('Word Count by Sentiment')
axes[1].set_xlabel('Sentiment')
axes[1].set_ylabel('Word Count')
plt.sca(axes[1])
plt.title('Word Count by Sentiment')

# Text length histogram
for label, color in zip(df['sentiment_label'].unique(), colors):
    subset = df[df['sentiment_label'] == label]['text_length']
    axes[2].hist(subset, alpha=0.6, label=label, color=color, bins=25)
axes[2].set_title('Text Length Distribution')
axes[2].set_xlabel('Character Length')
axes[2].legend()

plt.suptitle('Dataset Analysis', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('results/dataset_analysis.png', dpi=150, bbox_inches='tight')
plt.show()


## Task 2: Text Preprocessing

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    """
    Full text cleaning pipeline:
    1. Lowercase
    2. Remove special characters / numbers
    3. Tokenize
    4. Remove stopwords
    5. Lemmatize
    """
    # 1. Lowercase
    text = text.lower()
    # 2. Remove special characters and numbers
    text = re.sub(r'[^a-z\s]', '', text)
    # 3. Tokenize
    tokens = word_tokenize(text)
    # 4. Remove stopwords & short tokens
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    # 5. Lemmatize
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

# Apply preprocessing
df['cleaned_message'] = df['customer_message'].apply(preprocess_text)

# Show before/after
print("Before:", df['customer_message'].iloc[0])
print("\nAfter :", df['cleaned_message'].iloc[0])
print(f"\nPreprocessing complete. {len(df)} records processed.")


## Task 3: Text Vectorization

### Why Must Text Be Converted to Vectors?

Machine learning models work with **numbers**, not raw text. Text must be transformed into numerical vectors so that:
- Mathematical operations (dot products, gradients) can be performed
- The model can learn patterns and relationships between words
- Distances and similarities between texts can be computed

We use three approaches here: **Bag of Words**, **TF-IDF**, and **Tokenizer Sequences** for the LSTM.


In [ ]:
# Encode target labels
label_map = {'positive': 2, 'neutral': 1, 'negative': 0}
df['label'] = df['sentiment_label'].map(label_map)

# Train/test split
X = df['cleaned_message']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples : {len(X_train)}")
print(f"Testing  samples : {len(X_test)}")


In [ ]:
# --- Bag of Words ---
bow_vectorizer = CountVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow  = bow_vectorizer.transform(X_test)
print(f"BoW matrix shape: {X_train_bow.shape}")

# --- TF-IDF ---
tfidf_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2),
                                    sublinear_tf=True)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf  = tfidf_vectorizer.transform(X_test)
print(f"TF-IDF matrix shape: {X_train_tfidf.shape}")

# Top TF-IDF terms per class
print("\n--- Top 10 TF-IDF Terms (overall) ---")
feature_names = tfidf_vectorizer.get_feature_names_out()
mean_tfidf = X_train_tfidf.mean(axis=0).A1
top_idx = mean_tfidf.argsort()[-10:][::-1]
print([feature_names[i] for i in top_idx])


## Task 4: Baseline Model

In [ ]:
# ── Model 1: Naive Bayes with Bag of Words ──
nb_model = MultinomialNB()
nb_model.fit(X_train_bow, y_train)
nb_preds = nb_model.predict(X_test_bow)
nb_acc   = accuracy_score(y_test, nb_preds)

print("=== Naive Bayes (Bag of Words) ===")
print(f"Accuracy: {nb_acc*100:.2f}%")
print(classification_report(y_test, nb_preds,
      target_names=['negative', 'neutral', 'positive']))


In [ ]:
# ── Model 2: Logistic Regression with TF-IDF ──
lr_model = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
lr_model.fit(X_train_tfidf, y_train)
lr_preds = lr_model.predict(X_test_tfidf)
lr_acc   = accuracy_score(y_test, lr_preds)

print("=== Logistic Regression (TF-IDF) ===")
print(f"Accuracy: {lr_acc*100:.2f}%")
print(classification_report(y_test, lr_preds,
      target_names=['negative', 'neutral', 'positive']))


In [ ]:
# Plot confusion matrices side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
class_names = ['Negative', 'Neutral', 'Positive']

for ax, preds, title in zip(axes,
    [nb_preds, lr_preds],
    ['Naive Bayes (BoW)', 'Logistic Regression (TF-IDF)']):
    cm = confusion_matrix(y_test, preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(title, fontsize=12)

plt.suptitle('Baseline Model Confusion Matrices', fontsize=14)
plt.tight_layout()
plt.savefig('results/baseline_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Summary comparison table
results = pd.DataFrame({
    'Model': ['Naive Bayes (BoW)', 'Logistic Regression (TF-IDF)'],
    'Vectorization': ['Bag of Words', 'TF-IDF'],
    'Accuracy (%)': [round(nb_acc*100, 2), round(lr_acc*100, 2)]
})
print(results.to_string(index=False))
results.to_csv('results/model_comparison.csv', index=False)


## Task 5: Sequence Model – LSTM

In [ ]:
# Tokenizer-based sequences for LSTM
VOCAB_SIZE  = 10000
MAX_LEN     = 50
EMBED_DIM   = 64

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

# Convert text → sequences → padded arrays
X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train),
                             maxlen=MAX_LEN, padding='post', truncating='post')
X_test_seq  = pad_sequences(tokenizer.texts_to_sequences(X_test),
                             maxlen=MAX_LEN, padding='post', truncating='post')

print(f"Vocabulary size     : {min(VOCAB_SIZE, len(tokenizer.word_index))}")
print(f"Training seq shape  : {X_train_seq.shape}")
print(f"Testing  seq shape  : {X_test_seq.shape}")
print(f"\nSample sequence (first 10 tokens): {X_train_seq[0][:10]}")


In [ ]:
# Build LSTM model
NUM_CLASSES = 3

lstm_model = keras.Sequential([
    # Embedding: converts token IDs → dense vectors
    layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM,
                     input_length=MAX_LEN),

    # Bidirectional LSTM: reads sequence forward AND backward
    layers.Bidirectional(layers.LSTM(64, return_sequences=True)),
    layers.Dropout(0.3),

    # Second LSTM layer
    layers.LSTM(32),
    layers.Dropout(0.3),

    # Classification head
    layers.Dense(64, activation='relu'),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

lstm_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

lstm_model.summary()


In [ ]:
# Train LSTM
early_stop = EarlyStopping(monitor='val_loss', patience=5,
                           restore_best_weights=True)

history = lstm_model.fit(
    X_train_seq, y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)


In [ ]:
# Evaluate LSTM
lstm_loss, lstm_acc = lstm_model.evaluate(X_test_seq, y_test, verbose=0)
print(f"LSTM Test Loss    : {lstm_loss:.4f}")
print(f"LSTM Test Accuracy: {lstm_acc*100:.2f}%")

lstm_preds = np.argmax(lstm_model.predict(X_test_seq, verbose=0), axis=1)
print("\n--- LSTM Classification Report ---")
print(classification_report(y_test, lstm_preds,
      target_names=['Negative', 'Neutral', 'Positive']))


In [ ]:
# Plot LSTM training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, title in zip(axes, ['accuracy', 'loss'], ['Accuracy', 'Loss']):
    ax.plot(history.history[metric],         label=f'Train', linewidth=2)
    ax.plot(history.history[f'val_{metric}'], label=f'Validation',
            linewidth=2, linestyle='--')
    ax.set_title(f'LSTM – {title}', fontsize=13)
    ax.set_xlabel('Epoch'); ax.set_ylabel(title)
    ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/lstm_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Final model comparison
all_results = pd.DataFrame({
    'Model': ['Naive Bayes (BoW)', 'Logistic Regression (TF-IDF)', 'Bidirectional LSTM'],
    'Type': ['Baseline', 'Baseline', 'Sequence Model'],
    'Accuracy (%)': [round(nb_acc*100, 2), round(lr_acc*100, 2), round(lstm_acc*100, 2)]
})

print("\n=== Final Model Comparison ===")
print(all_results.to_string(index=False))
all_results.to_csv('results/model_evaluation.csv', index=False)

# Bar chart
fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#3498db', '#2ecc71', '#e74c3c']
bars = ax.bar(all_results['Model'], all_results['Accuracy (%)'],
              color=colors, edgecolor='black', width=0.5)
for bar, val in zip(bars, all_results['Accuracy (%)']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val}%', ha='center', fontweight='bold')
ax.set_title('Model Accuracy Comparison', fontsize=14)
ax.set_ylabel('Accuracy (%)')
ax.set_ylim(0, 105)
ax.set_xticklabels(all_results['Model'], rotation=10, ha='right')
plt.tight_layout()
plt.savefig('results/model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Sample predictions output
label_map_inv = {2: 'positive', 1: 'neutral', 0: 'negative'}

sample_texts = X_test.iloc[:10].values
true_labels  = y_test.iloc[:10].values
sample_seqs  = X_test_seq[:10]
pred_labels  = np.argmax(lstm_model.predict(sample_seqs, verbose=0), axis=1)

lines = ["SAMPLE PREDICTIONS (LSTM Model)", "=" * 60]
for i, (text, true, pred) in enumerate(zip(sample_texts, true_labels, pred_labels)):
    status = "✓ CORRECT" if true == pred else "✗ WRONG"
    lines.append(f"\n[{i+1}] {status}")
    lines.append(f"Text : {text[:80]}...")
    lines.append(f"True : {label_map_inv[true]}")
    lines.append(f"Pred : {label_map_inv[pred]}")

output = '\n'.join(lines)
print(output)

with open('results/sample_predictions.txt', 'w') as f:
    f.write(output)
print("\nSaved: results/sample_predictions.txt")


## Task 6: Attention and Transformer Reflection

### Why RNNs Struggle with Long-Term Dependencies
RNNs process sequences **one token at a time**, passing a fixed-size hidden state forward. As sequences grow longer, earlier information is progressively overwritten by newer inputs — the hidden state essentially "forgets" distant context. This is called the **vanishing gradient problem**: gradients shrink exponentially as they backpropagate through many time steps, making it nearly impossible for the model to learn connections between words that are far apart.

### How LSTMs Help with Memory
LSTMs introduce three **gating mechanisms**:
- **Forget gate** — decides what old information to discard
- **Input gate** — decides what new information to store
- **Output gate** — decides what to output at each step

A separate **cell state** acts as a long-term memory highway that flows through the sequence with minimal modification. This allows LSTMs to selectively remember information from many steps earlier, solving the vanishing gradient problem for moderate sequence lengths.

### What Attention Solves
Attention allows a model to **directly look at any part of the input sequence** when producing each output token, instead of relying on a compressed hidden state. In sequence-to-sequence tasks (e.g., translation), the decoder can "attend" to the most relevant encoder positions — so translating the word "bank" in a long sentence can directly reference the word "river" from the beginning, regardless of distance. This eliminates the information bottleneck of fixed-size hidden states.

### Why Transformers Are Important in Modern NLP and Generative AI
Transformers replace recurrence entirely with **self-attention**, where every token attends to every other token in parallel. This gives three major advantages:
1. **Parallelism** — no sequential dependency, so training is dramatically faster on GPUs
2. **Global context** — every word has direct access to every other word from the first layer
3. **Scale** — transformers scale efficiently to billions of parameters

Models like **GPT-4, BERT, LLaMA, and Claude** are all transformer-based. They power modern NLP tasks — translation, summarization, Q&A, code generation, and generative AI — because transformers learn rich contextual representations of language at scale.
